# Kak Skilly — Kaggle inference + ingest

Runs **vLLM (Qwen3-VL-8B)** behind a shared-secret auth proxy, a **heartbeat** to Supabase, the **PDF/HTML ingest loop**, and a **Cloudflare named tunnel** that the Worker reaches as `KAGGLE_BASE_URL`.

Settings: Accelerator **GPU T4 x2**, Internet **on**. Session limit 12 h; weekly GPU quota ~30 h — the Worker falls back to Workers AI when this is off.

Required **Kaggle Secrets** (Add-ons → Secrets): `SKILLY_SHARED_SECRET`, `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `CF_ACCOUNT_ID`, `CF_API_TOKEN`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `TELEGRAM_BOT_TOKEN`, `CF_TUNNEL_TOKEN`, `REPO_URL`.

See `docs/telegram-rag-learning-companion/setup.md`.

In [ ]:
# 1. Load secrets into the environment (never print them)
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for k in ['SKILLY_SHARED_SECRET','SUPABASE_URL','SUPABASE_SERVICE_KEY','CF_ACCOUNT_ID','CF_API_TOKEN',
          'R2_ACCESS_KEY_ID','R2_SECRET_ACCESS_KEY','TELEGRAM_BOT_TOKEN','CF_TUNNEL_TOKEN','REPO_URL']:
    os.environ[k] = _s.get_secret(k)
os.environ.setdefault('SKILLY_MODEL', 'Qwen/Qwen3-VL-8B-Instruct-AWQ')
print('secrets loaded')

In [ ]:
# 2. Install. PIN VERSIONS before first run: choose a vLLM release with Qwen3-VL support that is at least
#    two weeks old, and a cloudflared release likewise. Replace the placeholders below.
VLLM_VERSION = 'REPLACE_ME'        # e.g. a 0.11.x+ release
CLOUDFLARED_VERSION = 'REPLACE_ME' # e.g. 2025.x.y
assert 'REPLACE_ME' not in (VLLM_VERSION, CLOUDFLARED_VERSION), 'pin versions first'
!pip install -q vllm=={VLLM_VERSION}
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!git clone --depth 1 "$REPO_URL" /kaggle/working/skilly
!pip install -q -e /kaggle/working/skilly/ingest -e /kaggle/working/skilly/kaggle

In [ ]:
# 3. Start services in the background; logs go to /kaggle/working/*.log (no prompts are logged)
import subprocess
procs = {
  'serve':   subprocess.Popen(['python','-m','skilly_kaggle.serve'], stdout=open('/kaggle/working/serve.log','w'), stderr=subprocess.STDOUT),
  'ingest':  subprocess.Popen(['python','-m','skilly_kaggle.ingest_loop'], stdout=open('/kaggle/working/ingest.log','w'), stderr=subprocess.STDOUT),
  'tunnel':  subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','run','--token',os.environ['CF_TUNNEL_TOKEN']], stdout=open('/kaggle/working/tunnel.log','w'), stderr=subprocess.STDOUT),
}
print({k: p.pid for k, p in procs.items()})

In [ ]:
# 4. Keep the session alive and show health every minute. Stop this cell to shut down.
import time, urllib.request
while True:
    try:
        ok = urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5).status == 200
    except Exception:
        ok = False
    print(time.strftime('%H:%M:%S'), 'vLLM healthy' if ok else 'vLLM not ready', {k: p.poll() for k, p in procs.items()})
    time.sleep(60)